In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os

In [2]:
load_dotenv(override=True)

client = Anthropic()
model = "claude-sonnet-5"

In [4]:
# Scope tools to specific agent needs - keep tool availablity to a minimum per agent
main_support_agent_tools = [
    "get_customer_by_email",
    "lookup_order_by_id",
    "check_refund_eligibility",
    "create_human_escalation",
]

refund_agent_tools = [
    "lookup_order_by_id",
    "check_refund_eligibility",
    "process_refund",
    "create_human_escalation",
]

In [5]:
CUSTOMERS = {
    "alex@example.com": {
        "customer_id": "CUS-1001",
        "email": "alex@example.com",
        "name": "Alex Morgan",
        "account_status": "active"
    }
}

ORDERS = {
    "ORD-12345678": {
        "order_id": "ORD-12345678",
        "customer_id": "CUS-1001",
        "status": "delivered",
        "delivered_days_ago": 12,
        "total": 89.99,
        "currency": "USD",
    },
    "ORD-12345678": {
        "order_id": "ORD-87654321",
        "customer_id": "CUS-1001",
        "status": "delivered",
        "delivered_days_ago": 45,
        "total": 149.99,
        "currency": "USD",
    },
}

In [6]:
def get_customer_by_email(email: str) -> dict:
    normalized_email = email.strip().lower()

    customer = CUSTOMERS.get(normalized_email)

    if not customer:
        return {
            "isError": False,
            "customer": None,
            "message": "No customer was found for this email address.",
        }

    return {
        "isError": False,
        "customer": customer,
    }

In [9]:
def lookup_order_by_id(order_id: str, customer_id: str) -> dict:
    normalized_order_id = order_id.strip().upper()

    if not normalized_order_id.startswith("ORD-"):
        return {
            "isError": True,
            "errorCategory": "validation",
            "isRetryable": False,
            "customerMessage": "The order ID does not look valid.  Please check the order number and try again.  Orders start with 'ORD-'...",
            "developerMessage": "Order ID must start with 'ORD-'",
        }

    order = ORDERS.get(normalized_order_id)

    if not order:
        return {
            "isError": False,
            "order": None,
            "message": "No order was found with this order ID."
        }

    if order["customer_id"] != customer_id:
        return {
            "isError": True,
            "errorCategory": "permission",
            "isRetryable": False,
            "customerMessage": "I can't access this order with the current account information.",
            "developerMessage": "Order does not belong to the authenticated customer."
        }

    return {
        "isError": False,
        "order": order
    }


In [ ]:
def check_refunc_eligibility(order: dict) -> dict:
    if order["status"] != "delivered":
        return {
            "isError": True,
            "errorCategory": "business",
            "isRetryable": False,
            "customerMessage": "This order is not eligible for an automatic refund because it has not been delivered.",
            "developerMessage": "Refund denied because order status is not delivered",
        }

    if order["delivered_days_ago"] > 30:
        return {
            "isError": True,
            "errorCategory": "business",
            "isRetryable": False,
            "customerMessage": "This order is outside the 30-day return window, so I cannot process this request automatically.",
            "developerMessage": "Refund denied delivery date is outside the 30-day policy window.",
        }

    return {
        "isError": False,
        "eligible": True,
        "reason": "Order is within the 30-day return window."
    }

In [12]:
def process_refund(order_id: str, amount: float, reason: str) -> dict:
    if amount <= 0:
        return {
            "isError": True,
            "errorCategory": "validation",
            "isRetryable": False,
            "customerMessage": "This refund amount must be greater than zero.",
            "developerMessage": "Refund amount was less than or equal to zero.",
        }

    return {
            "isError": False,
            "refund": {
                "refund_id": "REF-555000",
                "order_id": order_id,
                "amount": amount,
                "status": "submitted",
                "reason": reason,
            },
        }
    